# ML-Based Ad Blocking — Random Forest, Logistic Regression, SVM & XGBoost
### Google Colab practical lab · Scikit-learn Pipelines · Hyperparameter tuning · Soft Voting

**Dataset:** `ad_urls (1).csv` with `url` and `label` columns (`0 = ALLOW`, `1 = BLOCK`).

**Workflow:** Upload CSV → inspect labels → holdout split → leakage-safe text Pipelines → tune four classifiers with five-fold CV → four-model Voting ensemble → select by **training CV** → report all holdout results → example predictions → save outputs.

> This experiment classifies URL strings. **It does not intercept web traffic or block ads in a browser.** Thirty synthetic URLs are not a realistic evaluation dataset.

## Step 1 — Install the required libraries
Run this cell first in Colab; XGBoost and Scikit-learn are installed if needed.

In [ ]:
%pip -q install "xgboost>=2.0" "scikit-learn>=1.4" pandas numpy matplotlib joblib

## Step 2 — Upload and read your ad URL CSV

The first code cell **reads the CSV with Pandas**. On Colab, choose `ad_urls (1).csv` if a matching file is not already in the working folder.

In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('ad_urls (1).csv'),
    Path('ad_urls.csv'),
    Path('data/ad_urls.csv'),
    Path('/content/ad_urls (1).csv'),
    Path('/content/ad_urls.csv'),
]
csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Place the CSV in this folder or run in Google Colab to upload it.') from exc
    uploaded = files.upload()  # Choose ad_urls (1).csv
    csv_path = next((Path(name) for name in uploaded if name.lower().endswith('.csv')), None)
    if csv_path is None:
        raise ValueError('Please upload a CSV file containing url and label columns.')

raw = pd.read_csv(csv_path)
print('Dataset file:', csv_path)
print('Shape:', raw.shape)
display(raw.head())

## Step 3 — Explore the dataset and validate the target

URL strings are the features. `label` is the target: **0 = ALLOW**, **1 = BLOCK**. Duplicate URL strings must not appear in both the training and test sets.

In [ ]:
print('Column names:', raw.columns.tolist())
print('Missing values:', raw.isna().sum().to_dict())
print('Class distribution:')
print(raw['label'].value_counts().sort_index())
print('\nRandom URL examples:')
display(raw.sample(min(5, len(raw)), random_state=42))

## Step 4 — Import ML libraries; clean and split the CSV

The **30% held-out test set** remains untouched during hyperparameter tuning and model selection. All text-vectorizer fitting occurs inside the Pipelines on training folds.

In [ ]:
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score,
    classification_report, f1_score, precision_score, recall_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_val_score, train_test_split,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import SVC
from xgboost import XGBClassifier

SEED = 42
LABELS = {0: 'ALLOW', 1: 'BLOCK'}

def load_urls(path: str | Path):
    """Read CSV; remove invalid rows, contradictory and duplicate URLs."""
    df = pd.read_csv(path)
    if not {'url', 'label'}.issubset(df.columns):
        raise ValueError(f"CSV requires 'url' and 'label' columns, got {list(df.columns)}")
    df = df[['url', 'label']].dropna().copy()
    df['url'] = df['url'].astype(str).str.strip()
    df['label'] = pd.to_numeric(df['label'], errors='raise').astype(int)
    df = df[df['url'] != '']
    if not set(df['label']).issubset({0, 1}) or df['label'].nunique() != 2:
        raise ValueError('Expected both binary labels: 0=ALLOW and 1=BLOCK')
    contradictions = df.groupby('url')['label'].nunique()
    if (contradictions > 1).any():
        raise ValueError('The same URL occurs with conflicting labels')
    df = df.drop_duplicates(subset=['url']).reset_index(drop=True)
    if df['label'].value_counts().min() < 6:
        raise ValueError('Each class must have at least 6 examples for a 5-fold training CV')
    print(f"Dataset: {len(df)} URLs; label counts {df['label'].value_counts().to_dict()}")
    print('0 = ALLOW | 1 = BLOCK')
    return df

df = load_urls(csv_path)
X_train, X_test, y_train, y_test = train_test_split(
    df['url'], df['label'], test_size=0.30,
    random_state=SEED, stratify=df['label'],
)
print(f'Training samples: {len(X_train)}; independent test samples: {len(X_test)}')

## Step 5 — Construct four Scikit-learn Pipelines

All four Pipelines contain the same **FeatureUnion** before their classifier:

- Character TF–IDF (3–5-grams): parts of URLs, including domains and paths.
- Token TF–IDF (1–2-grams): text tokens and token pairs from URLs.
- A small fixed vocabulary of common ad/tracker terms, given increased weight. The list is domain knowledge, not learned from the held-out labels.

**Models:** Random Forest, Logistic Regression, SVM, XGBoost. The vocabulary learns from each training fold, avoiding leakage.

In [ ]:
AD_TERMS = [
    'ad', 'ads', 'adserver', 'advertisement', 'advertising', 'banner',
    'doubleclick', 'tracker', 'tracking', 'analytics', 'pixel', 'collect',
    'retargeting', 'impression', 'sponsor', 'promoted', 'affiliate',
    'marketing', 'campaign', 'beacon',
]

def make_pipeline(model):
    """All TF-IDF vocabulary construction is FIT ONLY on training folds."""
    extractor = FeatureUnion([
        ('characters', TfidfVectorizer(
            analyzer='char', ngram_range=(3, 5), sublinear_tf=True)),
        ('tokens', TfidfVectorizer(
            token_pattern=r'(?u)[a-zA-Z]{2,}', ngram_range=(1, 2),
            sublinear_tf=True)),
        ('ad_terms', CountVectorizer(
            token_pattern=r'(?u)[a-zA-Z]{2,}', vocabulary=AD_TERMS, binary=True)),
    ], transformer_weights={'characters': 1.0, 'tokens': 1.0, 'ad_terms': 3.0})
    return Pipeline([('features', extractor), ('model', model)])

def make_models():
    return {
        'Random Forest': make_pipeline(RandomForestClassifier(
            n_estimators=120, random_state=SEED, n_jobs=1)),
        'Logistic Regression': make_pipeline(LogisticRegression(
            max_iter=2000, random_state=SEED)),
        'SVM': make_pipeline(SVC(
            kernel='linear', probability=True, random_state=SEED)),
        'XGBoost': make_pipeline(XGBClassifier(
            objective='binary:logistic', eval_metric='logloss',
            tree_method='hist', random_state=SEED, n_jobs=1)),
    }

for name, pipe in make_models().items():
    print(f'{name}: {list(pipe.named_steps)}')
print('\nExample Pipeline:')
print(make_models()['SVM'])

## Step 6 — Tune each classifier using five-fold training cross-validation

`RandomizedSearchCV` tries **four configurations per model** to keep the Colab runtime practical. The four tuned models are also combined in an **equal-weight Soft VotingClassifier**. The Voting ensemble is not assumed to outperform SVM; its cross-validation accuracy is measured.

The **highest training CV score** determines which model is saved. The test set is never used to choose hyperparameters or the winning classifier.

In [ ]:
SEARCH_SPACES = {
    'Random Forest': {
        'model__n_estimators': [80, 120, 200],
        'model__max_depth': [None, 3, 6],
        'model__min_samples_leaf': [1, 2],
    },
    'Logistic Regression': {
        'model__C': [0.1, 1.0, 10.0, 100.0],
        'model__class_weight': [None, 'balanced'],
    },
    'SVM': {
        'model__kernel': ['linear', 'rbf'],
        'model__C': [0.1, 1.0, 10.0],
        'model__class_weight': [None, 'balanced'],
    },
    'XGBoost': {
        'model__n_estimators': [50, 100, 180],
        'model__max_depth': [2, 3, 4],
        'model__learning_rate': [0.03, 0.1, 0.2],
        'model__reg_lambda': [1, 5],
    },
}

def train_and_compare(X_train, y_train, n_iter=4):
    """Choose a model using training-only CV (never the held-out test set)."""
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    tuned = {}
    cv_rows = []
    for name, pipeline in make_models().items():
        print(f'\nTuning {name} with 5-fold CV ...', flush=True)
        search = RandomizedSearchCV(
            estimator=pipeline, param_distributions=SEARCH_SPACES[name],
            n_iter=n_iter, cv=folds, scoring='accuracy', n_jobs=1,
            random_state=SEED, refit=True, error_score='raise',
        )
        search.fit(X_train, y_train)
        tuned[name] = search.best_estimator_
        cv_rows.append({
            'model': name, 'cv_accuracy': float(search.best_score_),
            'parameters': json.dumps(search.best_params_),
        })
        print(f"  CV accuracy: {search.best_score_:.1%}; {search.best_params_}")

    voting = VotingClassifier(
        estimators=[
            ('rf', clone(tuned['Random Forest'])),
            ('lr', clone(tuned['Logistic Regression'])),
            ('svm', clone(tuned['SVM'])),
            ('xgb', clone(tuned['XGBoost'])),
        ], voting='soft', n_jobs=1,
    )
    print('\nEvaluating four-model Soft Voting via training-only CV ...', flush=True)
    vote_scores = cross_val_score(
        voting, X_train, y_train, scoring='accuracy', cv=folds, n_jobs=1,
    )
    cv_rows.append({
        'model': 'Soft Voting (all four)', 'cv_accuracy': float(vote_scores.mean()),
        'parameters': 'equal-weight soft vote of four tuned Pipelines',
    })
    cv_table = pd.DataFrame(cv_rows).sort_values(
        ['cv_accuracy', 'model'], ascending=[False, True],
    ).reset_index(drop=True)
    winner = cv_table.iloc[0]['model']
    print('\nTRAINING CV COMPARISON:')
    print(cv_table[['model', 'cv_accuracy']].to_string(index=False))
    print('Selected by training CV:', winner)
    return {**tuned, 'Soft Voting (all four)': voting}, cv_table, winner

candidates, cv_table, winner = train_and_compare(X_train, y_train, n_iter=4)
display(cv_table[['model', 'cv_accuracy', 'parameters']])

## Step 7 — Visualize the training CV comparison

Use this chart to explain model selection without looking at the held-out scores.

In [ ]:
ax = cv_table.sort_values('cv_accuracy').plot.barh(
    x='model', y='cv_accuracy', legend=False, figsize=(9, 4.3)
)
ax.set_xlabel('Training 5-fold CV accuracy')
ax.set_ylabel('')
ax.set_xlim(0, 1.05)
ax.set_title('URL classification — selection based on training CV')
plt.tight_layout()
plt.show()

## Step 8 — Evaluate all candidates on the held-out test set

After CV selects the winner, this cell scores **all five candidates for transparent comparison**, but does **not** choose a different winner using the test results.

Metrics include **accuracy, balanced accuracy, precision and recall of the BLOCK class, macro-F1, confusion matrix**, and the majority-class baseline. Metrics, predictions and the CV-selected fitted Pipeline are saved to `outputs/ad_blocker/`.

In [ ]:
def evaluate_and_save(
    X_train, X_test, y_train, y_test, candidates, cv_table,
    winner, output_dir='outputs/ad_blocker', demo_urls=None,
):
    """Report ALL holdout scores for transparency, save only CV-selected model."""
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    baseline = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
    baseline_accuracy = float(accuracy_score(y_test, baseline.predict(X_test)))
    rows = []
    fitted_winner = None
    winner_preds = None
    for name, candidate in candidates.items():
        fitted = clone(candidate).fit(X_train, y_train)
        pred = fitted.predict(X_test)
        rows.append({
            'model': name, 'test_accuracy': float(accuracy_score(y_test, pred)),
            'balanced_accuracy': float(balanced_accuracy_score(y_test, pred)),
            'block_precision': float(precision_score(y_test, pred, zero_division=0)),
            'block_recall': float(recall_score(y_test, pred, zero_division=0)),
            'macro_f1': float(f1_score(y_test, pred, average='macro', zero_division=0)),
        })
        if name == winner:
            fitted_winner, winner_preds = fitted, pred
    test_table = pd.DataFrame(rows).sort_values('test_accuracy', ascending=False)
    winner_accuracy = float(accuracy_score(y_test, winner_preds))
    print('\nHOLDOUT MODEL COMPARISON (NOT used for winner selection):')
    print(test_table.to_string(index=False, float_format=lambda val: f'{val:.3f}'))
    print(f'\nMajority-class baseline: {baseline_accuracy:.1%}')
    print(f'CV-selected model: {winner}; test accuracy: {winner_accuracy:.1%} '
          f'({int((winner_preds == y_test).sum())}/{len(y_test)} correct)')
    print('\nClassification report for CV-selected model:')
    print(classification_report(
        y_test, winner_preds, labels=[0, 1],
        target_names=['ALLOW', 'BLOCK'], digits=3, zero_division=0,
    ))
    disp = ConfusionMatrixDisplay.from_predictions(
        y_test, winner_preds, labels=[0, 1],
        display_labels=['ALLOW', 'BLOCK'], cmap='Blues', colorbar=False,
    )
    disp.ax_.set_title(f'CV-selected model: {winner}')
    plt.tight_layout()
    plt.savefig(output / 'ad_confusion_matrix.png', dpi=160)
    plt.close()

    cv_table.to_csv(output / 'ad_cv_scores.csv', index=False)
    test_table.to_csv(output / 'ad_holdout_scores.csv', index=False)
    pd.DataFrame({
        'url': X_test.to_numpy(), 'actual_label': y_test.to_numpy(),
        'predicted_label': winner_preds,
    }).to_csv(output / 'ad_holdout_predictions.csv', index=False)
    model_file = output / 'ad_best_pipeline.joblib'
    joblib.dump(fitted_winner, model_file)
    result = {
        'chosen_by_training_cv': winner, 'heldout_accuracy': winner_accuracy,
        'heldout_baseline': baseline_accuracy,
        'training_rows': len(X_train), 'test_rows': len(X_test),
        'correct_test_predictions': int((winner_preds == y_test).sum()),
        'note': 'Toy example: 30 synthetic URLs; results are not generalizable',
    }
    (output / 'ad_metrics.json').write_text(
        json.dumps(result, indent=2), encoding='utf-8',
    )
    print('\nExample URL classifications (demo, NOT live network blocking):')
    demo_urls = demo_urls or [
        'https://ads.example.net/banner/latest.js',
        'https://news.example.org/science',
        'https://tracker.example.com/pixel?id=9',
    ]
    for url in demo_urls:
        p = float(fitted_winner.predict_proba([url])[0, 1])
        label = LABELS[int(fitted_winner.predict([url])[0])]
        print(f'  {label:5s} | score={p:.3f} | {url}')
    print(f'\nSaved: {output}/')
    print('CAUTION: data are synthetic and far too small for practical ad blocking.')
    return result, test_table

results, test_table = evaluate_and_save(
    X_train, X_test, y_train, y_test, candidates, cv_table,
    winner, output_dir='outputs/ad_blocker',
)
display(test_table)
from IPython.display import Image
display(Image(filename='outputs/ad_blocker/ad_confusion_matrix.png'))

## Step 9 — Classify a URL using the saved Pipeline

The fitted Pipeline returns `ALLOW` or `BLOCK` plus a probability-like score. Predictions are a **demonstration only** (no real ad-blocking is performed). Choose URLs different from the training file.

In [ ]:
model = joblib.load('outputs/ad_blocker/ad_best_pipeline.joblib')
new_urls = [
    'https://ads.example.net/banner/square.js',
    'https://university.edu/library/books',
    'https://tracker.example.com/pixel?event=click',
]
rows = []
for url in new_urls:
    decision = int(model.predict([url])[0])
    block_score = float(model.predict_proba([url])[0, list(model.classes_).index(1)])
    rows.append({'url': url, 'prediction': LABELS[decision], 'block_probability': round(block_score, 4)})
display(pd.DataFrame(rows))

## Step 10 — Download experiment outputs

The ZIP includes CV scores, holdout scores, predictions, the confusion matrix, a JSON metrics file and the fitted Pipeline (joblib). Only **load model files you trust**.

In [ ]:
import shutil
zip_path = shutil.make_archive('ad_blocker_four_model_results', 'zip', 'outputs/ad_blocker')
print('Created:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Open the ZIP from your local working directory.')

## Step 11 — Reproduce and document the results on GitHub

- GitHub: **[Ruchika-choudhary8/new](https://github.com/Ruchika-choudhary8/new)** (check repository visibility before publishing sensitive data).
- Files: this notebook, `ad_blocker_four_models.py`, `README_ad_blocker_four_models.md`, `requirements.txt`, `data/ad_urls.csv`.
- Run `python ad_blocker_four_models.py --csv "ad_urls (1).csv"` for the equivalent command-line workflow.
- **Do not claim real-world 100% accuracy:** 30 toy URLs and 9 test cases are insufficient for a defensible field evaluation. Test on new domains, balanced modern blocklist/allowlist data and false-positive/false-negative costs.
- No browser integration is provided. A practical blocker needs a browser extension or network filtering integration, privacy review, safe allowlist controls, and systematic testing.